# Cheminformatics Workflow

A typical drug-discovery workflow: start with a CSV of SMILES, build a TMAP,
and explore structure-activity relationships with interactive coloring.

This notebook covers:

1. Loading molecules from a CSV
2. Computing fingerprints with `fingerprints_from_smiles`
3. Computing molecular properties with `molecular_properties`
4. Fitting a TMAP
5. Bulk-adding metadata with `add_metadata`
6. Rendering molecule structures in tooltips with `add_smiles`

**Requirements:** `pip install tmap rdkit`

## 1. Load data

We use a 2,000-molecule sample from a ChEMBL cluster. In practice you
would load your own CSV (or DataFrame from a database query, SDF file, etc.).

In [ ]:
import numpy as np
import pandas as pd

# Load SMILES from CSV (just the first 2000 for speed)
df = pd.read_csv("../examples/cluster_65053.csv", nrows=2000)
smiles = df["smiles"].tolist()

print(f"{len(smiles)} molecules loaded")
print(f"Example: {smiles[0]}")

## 2. Compute fingerprints

`fingerprints_from_smiles` wraps RDKit's Morgan fingerprint generator with
parallel processing. It returns a binary matrix and the indices of valid
molecules (invalid SMILES are dropped silently).

In [ ]:
from tmap.utils import fingerprints_from_smiles

fps, valid_idx = fingerprints_from_smiles(smiles, fp_type="morgan", radius=2, n_bits=2048)

# Keep only valid molecules
smiles = [smiles[i] for i in valid_idx]

print(f"Fingerprints: {fps.shape}  (valid: {len(valid_idx)}/{len(df)})")

## 3. Compute molecular properties

`molecular_properties` computes MW, LogP, and ring count for each SMILES.
Returns a dict of arrays — perfect for `add_metadata` later.

In [ ]:
from tmap.utils import molecular_properties

props = molecular_properties(smiles)

print(f"MW range:     {np.nanmin(props['mw']):.0f} – {np.nanmax(props['mw']):.0f} Da")
print(f"LogP range:   {np.nanmin(props['logp']):.1f} – {np.nanmax(props['logp']):.1f}")
print(f"Ring count:   {int(np.nanmin(props['n_rings']))} – {int(np.nanmax(props['n_rings']))}")

## 4. Fit TMAP

Binary fingerprints → Jaccard metric (the default). Three lines.

In [ ]:
from tmap import TMAP

model = TMAP(n_neighbors=20, seed=42).fit(fps)

print(f"Embedding:  {model.embedding_.shape}")
print(f"Tree edges: {model.tree_.edges.shape[0]}")

## 5. Visualize with `add_metadata`

Instead of calling `add_color_layout` once per column, pass a dict
(or DataFrame) to `add_metadata`. It auto-detects continuous vs. categorical:

- **Numeric with many unique values** → continuous (viridis colormap)
- **Numeric with few unique values** (≤20) → categorical (tab10)
- **Strings** → categorical

Here we pass the molecular properties dict directly.

In [ ]:
viz = model.to_tmapviz()
viz.title = "Chemical Space — Morgan FP + Jaccard"

# Bulk-add all molecular properties as color layers
viz.add_metadata(props)

# MW and LogP have many unique values → continuous (auto-detected)
# n_rings has few unique values → categorical (auto-detected)
print(f"Layouts added: {[l.name for l in viz.layouts]}")

### Add SMILES tooltips

With `add_smiles`, hovering over a point in the HTML visualization
renders the 2D molecular structure (via RDKit depiction).

In [ ]:
viz.add_smiles(smiles)

### Export & display

Write a self-contained HTML file (opens in any browser), or display
inline in Jupyter.

In [ ]:
# HTML export (shareable)
path = viz.write_html("cheminformatics_tmap")
print(f"Saved: {path}")

# Jupyter inline (interactive)
viz.show()

## 6. Working with DataFrames

`add_metadata` also accepts a pandas DataFrame. This is convenient when
your metadata lives in a table with mixed column types.

In [ ]:
# Build a DataFrame with properties + custom annotations
meta_df = pd.DataFrame({
    "MW": props["mw"],
    "LogP": props["logp"],
    "Lipinski": ["pass" if mw < 500 and lp < 5 else "fail"
                 for mw, lp in zip(props["mw"], props["logp"])],
})

viz2 = model.to_tmapviz()
viz2.title = "Lipinski Filter"

# One line adds all columns — MW and LogP as continuous, Lipinski as categorical
viz2.add_metadata(meta_df)
viz2.add_smiles(smiles)

viz2.show()

## 7. Tree exploration — activity gradients

Even without bioactivity data, the tree reveals structural gradients.
Here we compute pseudotime from a low-MW molecule and trace the MW
gradient along the tree.

In [ ]:
# Find the lightest molecule as root
root = int(np.nanargmin(props["mw"]))
print(f"Root: index {root}, MW = {props['mw'][root]:.0f} Da")
print(f"SMILES: {smiles[root]}")

# Pseudotime = tree distance from root
pseudotime = model.distances_from(root)

# Path from lightest to heaviest
heaviest = int(np.nanargmax(props["mw"]))
path_nodes = model.path(root, heaviest)
print(f"\nPath from lightest to heaviest: {len(path_nodes)} hops")
print(f"MW along path: {props['mw'][root]:.0f} → {props['mw'][heaviest]:.0f} Da")

model.plot_static(color_by=pseudotime, color_map="magma", point_size=4)

## 8. Selecting columns

If your DataFrame has many columns but you only want a few in the
visualization, use the `columns` parameter.

In [ ]:
big_df = pd.DataFrame({
    "MW": props["mw"],
    "LogP": props["logp"],
    "n_rings": props["n_rings"],
    "internal_id": range(len(smiles)),     # not useful for coloring
    "batch_number": np.ones(len(smiles)),  # constant, not useful
})

viz3 = model.to_tmapviz()
viz3.add_metadata(big_df, columns=["MW", "LogP"])  # only MW and LogP

print(f"Layouts: {[l.name for l in viz3.layouts]}")
# internal_id and batch_number were excluded

## Summary

| Step | Code |
|------|------|
| Fingerprints | `fps, idx = fingerprints_from_smiles(smiles)` |
| Properties | `props = molecular_properties(smiles)` |
| Fit TMAP | `model = TMAP().fit(fps)` |
| Add all metadata | `viz.add_metadata(props)` or `viz.add_metadata(df)` |
| Add molecules | `viz.add_smiles(smiles)` |
| Export | `viz.write_html("output")` |
| Select columns | `viz.add_metadata(df, columns=["MW", "LogP"])` |

`add_metadata` auto-detects column types. For manual control, use
`add_color_layout(name, values, categorical=True/False)` directly.